# 02 · Build the vector store

Embed the prepared documents with OpenAI and persist them in a local Chroma database (`db/chroma/`). Rebuilding resets the collection, so it never duplicates documents.

Requires `OPENAI_API_KEY` (environment or `.env`) and the output of `01_preprocess.ipynb`.

In [ ]:
import sys
from pathlib import Path

# Make the project root importable when running from notebooks/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

In [ ]:
from src.config import DOCUMENTS_FILE, EMBED_MODEL, require_api_key
from src.data_loader import load_documents
from src.vectorstore import build_vectorstore

require_api_key()
docs = load_documents(DOCUMENTS_FILE)
print(f"{len(docs)} documents loaded; embedding with {EMBED_MODEL}")

In [ ]:
vs = build_vectorstore(docs)
print("Documents in collection:", len(vs.get()["ids"]))

## Retrieval check

Lower distance means more similar.

In [ ]:
probes = [
    "childhood environmental exposures",
    "psychological stress and cortisol",
    "alcohol, obesity and physical activity",
]

for q in probes:
    print(f"\n> {q}")
    for doc, score in vs.similarity_search_with_score(q, k=3):
        print(f"  {score:.3f}  {doc.metadata['type']:<8}  {doc.metadata['source'][:70]}")